# Hyperparameter Grid Search for Adaptive Filters: cvx_comb_sKF

This notebook tests and demonstrates the generic `grid_search` function implemented in `filters_exp.py`.

### Objectives
1. **Generic Hyperparameter Search**: Separate algorithm hyperparameters into fixed performance parameters (controlling steady-state MSD/EMSE) and a multi-dimensional grid of convergence hyperparameters.
2. **Algorithm under Test**: `cvx_comb_sKF` (convex combination of fast and slow scaled Kalman filters).
3. **Search Space**:
   - Fixed steady-state performance parameters: $\epsilon_2$, $\sigma_\eta^2$, $\tilde{v}_0$, $\tau$, $l$, $\rho_{reg}$, $\lambda_0$.
   - Convergence hyperparameters grid: $\mu_a$, $\epsilon_1$, and $a_+$ with $a_+ \in \{2, 3, 4, 5, 6\}$.
4. **Validation & Visualizations**:
   - Automated functional assertions verifying grid shapes, optimality, and steady-state invariance.
   - Comprehensive visualizations:
     - 2D Pairwise heatmaps at the global optimum across all hyperparameter pairs.
     - Steady-state invariance heatmaps verifying that steady-state MSD is unaffected by convergence parameters.
     - Comparative MSD learning curves (Optimal vs. Worst vs. Median vs. Constituent Filters).
     - Interactive heatmap with dropdowns and sliders for exploration.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from collections import namedtuple
import time

import filters
import metrics
import monte_carlo as mc
import filters_exp as exp
import visual_tools as vt

%matplotlib inline
%config InlineBackend.figure_format = 'svg'

print("Modules imported successfully.")

## 1. Simulation Environment & System Setup

We configure a system identification experiment with an unknown plant $h_o$, driven by a colored AR input signal and corrupted by additive Gaussian observation noise.

In [ ]:
# System parameters
L = 16
ho = np.sinc(np.linspace(0, 1, L))
ho = ho / np.linalg.norm(ho)
h0 = np.zeros(L, dtype=np.float64)

# Environment configuration
var_x = 1.0
var_v = 1e-3  # Gaussian noise variance (30 dB SNR)
AR = np.array([1.0, -0.6], dtype=np.float64)

env_parameters = filters.sys_inversion_gaussian_env_parameters(
    ho=ho, AR=AR, var_v=var_v, var_x=var_x
)
environment = filters.sys_inversion_gaussian_behavior

# Monte Carlo simulation settings
N = 50000
NR = 256
num_workers = 4
num_chunks = 8
seed = 42

print(f"Filter length L: {L}")
print(f"Iterations N: {N}, Realizations NR: {NR}")
print(f"Observation noise variance: {var_v} (SNR ~ 30 dB)")

## 2. Hyperparameter Specification: Performance vs. Convergence

We explicitly separate the parameters into:
- **Performance parameters** (fixed): $\epsilon_2 = 10^{-5}$ sets the steady-state MSD floor. Other fixed parameters are system noise variance estimate $\sigma_\eta^2$, initial variance $\tilde{v}_0$, combination memory $\tau$, weight transfer $l$, regularization $\rho_{reg}$, and transfer threshold $\lambda_0$.
- **Convergence parameters grid**:
  - $\mu_a$: Learning rate for combination weight adaptation.
  - $\epsilon_1$: Process noise variance for the fast filter (agility).
  - $a_+$: Asymmetry/interval constrain limit, with values in $\{2, 3, 4, 5, 6\}$.

In [ ]:
# Fixed performance parameters
cvx_perf_params = namedtuple('cvx_perf_params', [
    'epsilon_2', 'var_eta', 'v_tilde_0', 'tau', 'l', 'reg_rho', 'lambda_0'
])(
    epsilon_2=5e-9,
    var_eta=var_v,
    v_tilde_0=1.0,
    tau=1.0 - 10.0 / L,
    l=1.0 - 1.0 / L,
    reg_rho=1e-2,
    lambda_0=0.7
)

# Convergence parameters grid (namedtuple of NDArrays)
cvx_conv_grid = namedtuple('cvx_conv_grid', ['mu_a', 'epsilon_1', 'a_plus'])(
    mu_a=np.array([1.0, 2.5, 5.0, 7.5, 10.0], dtype=np.float64),
    epsilon_1=np.array([1e-6, 1e-5, 1e-4, 1e-3], dtype=np.float64),
    a_plus=np.array([5.0, 10.0, 15.0, 20.0, 25.0], dtype=np.float64)
)

print("Fixed performance parameters:")
for k, v in cvx_perf_params._asdict().items():
    print(f"  {k}: {v}")

print("\nConvergence parameter grid:")
for k, v in cvx_conv_grid._asdict().items():
    print(f"  {k}: {v} (length {len(v)})")

total_combinations = int(np.prod([len(v) for v in cvx_conv_grid]))
print(f"\nTotal grid combinations to evaluate: {total_combinations}")

## 3. Execution of `filters_exp.grid_search`

We run `grid_search` to find the optimal $(\mu_a^*, \epsilon_1^*, a_+^*)$ that minimizes the convergence metric (`db_area_of_convergence`) using Dask for parallel execution.

In [ ]:
t_start = time.time()

grid_results = exp.grid_search(
    N=N,
    NR=NR,
    h0=h0,
    environment=environment,
    environment_parameters=env_parameters,
    algorithm=filters.cvx_comb_sKF,
    algorithm_convergence_parameters=cvx_conv_grid,
    algorithm_performance_parameters=cvx_perf_params,
    performance_metric=metrics.squared_deviation,
    convergence_metric=exp.db_area_of_convergence,
    num_workers=num_workers,
    num_chunks=num_chunks,
    seed=seed,
    minimize_convergence_metric=True
)

t_elapsed = time.time() - t_start

print(f"Grid search completed in {t_elapsed:.2f} seconds.")
print("\nOptimal Hyperparameters Found:")
for k, v in grid_results['best_parameters'].items():
    print(f"  {k}*: {v}")
print(f"Best Convergence Metric (Area in dB*samples): {grid_results['best_metric_value']:.2f}")
print(f"Steady-State MSD at Optimal: {grid_results['best_steady_state']:.2f} dB")

## 4. Test Assertions & Functional Verification

We programmatically verify that:
1. All required keys exist in `grid_results`.
2. The multidimensional grid shapes match the grid dimensions `(4, 4, 5)`.
3. The selected best parameters indeed correspond to the minimum of `convergence_grid`.
4. Steady-state MSD is approximately constant across all tested configurations (confirming the steady-state performance separation hypothesis).

In [ ]:
# 1. Structure check
required_keys = [
    'best_parameters', 'best_metric_value', 'best_steady_state',
    'optimal_index', 'convergence_grid', 'steady_state_grid',
    'grid_axes', 'measures', 'best_label', 'labels_grid'
]
for key in required_keys:
    assert key in grid_results, f"Missing key in results: {key}"
print("✓ Return dictionary keys verified.")

# 2. Shape check
expected_shape = (len(cvx_conv_grid.mu_a), len(cvx_conv_grid.epsilon_1), len(cvx_conv_grid.a_plus))
assert grid_results['convergence_grid'].shape == expected_shape, f"Unexpected convergence grid shape: {grid_results['convergence_grid'].shape}"
assert grid_results['steady_state_grid'].shape == expected_shape, f"Unexpected steady-state grid shape: {grid_results['steady_state_grid'].shape}"
print(f"✓ Grid shapes verified: {expected_shape}.")

# 3. Optimality check
opt_idx = grid_results['optimal_index']
expected_best_val = grid_results['convergence_grid'][opt_idx]
assert np.isclose(expected_best_val, grid_results['best_metric_value']), "Optimal metric value mismatch"
assert grid_results['best_metric_value'] == np.nanmin(grid_results['convergence_grid']), "Optimal value is not the minimum"
print(f"✓ Optimality verified: min metric = {grid_results['best_metric_value']:.2f} at index {opt_idx}.")

# 4. Steady-state invariance check
ss_values = grid_results['steady_state_grid']
ss_std = float(np.nanstd(ss_values))
ss_mean = float(np.nanmean(ss_values))
print(f"Steady-state MSD across grid: Mean = {ss_mean:.2f} dB, Std Dev = {ss_std:.2f} dB")
assert ss_std < 1.0, f"Steady-state MSD varies significantly (std={ss_std:.2f} dB > 1.0 dB)"
print("✓ Steady-state invariance verified: std dev < 1.0 dB across all 80 combinations.")
print("\nALL FUNCTIONAL TESTS PASSED SUCCESSFULLY!")

## 5. Graphical Results: All Pairwise Heatmaps at Global Optimum

As requested, we plot all combinations of heatmaps for every pair of hyperparameters:
- For each 2D heatmap, the third hyperparameter not displayed on the axes is fixed at its **global optimal value**.
- The optimal parameter combination is highlighted on each map with a red star ($\star$).
- The subplot titles display the optimal values and the fixed parameter value.

In [ ]:
fig_conv = exp.plot_grid_search_pairwise_heatmaps(grid_results, metric_type='convergence')
plt.show()

### Steady-State Performance Heatmaps

Next, we inspect the measured steady-state MSD [dB] across the same pairwise slices, confirming visually that steady-state performance is uniform regardless of $\mu_a, \epsilon_1, a_+$.

In [ ]:
fig_ss = exp.plot_grid_search_pairwise_heatmaps(grid_results, metric_type='steady_state')
plt.show()

## 6. Learning Curves Comparison

We compare the convergence learning curve (MSD [dB] vs iteration) of:
1. The **Optimal configuration** found by `grid_search`.
2. The **Worst configuration** on the grid.
3. The **Median configuration**.
4. Standalone **Fast Filter** ($sKF_1$ with $\epsilon_1^*$).
5. Standalone **Slow Filter** ($sKF_2$ with $\epsilon_2$).

In [ ]:
# Extract best, worst, and median configurations
c_grid = grid_results['convergence_grid']
opt_idx = grid_results['optimal_index']
worst_flat = np.nanargmax(c_grid)
worst_idx = np.unravel_index(worst_flat, c_grid.shape)

valid_scores = c_grid[~np.isnan(c_grid)]
median_val = np.median(valid_scores)
median_flat = np.argmin(np.abs(c_grid - median_val))
median_idx = np.unravel_index(median_flat, c_grid.shape)

best_label = grid_results['labels_grid'][opt_idx]
worst_label = grid_results['labels_grid'][worst_idx]
median_label = grid_results['labels_grid'][median_idx]

# Run individual component filters for baseline comparison
fast_skf_param = filters.sKF_parameters(
    label='sKF_fast (Filter 1)',
    epsilon=grid_results['best_parameters']['epsilon_1'],
    var_eta=cvx_perf_params.var_eta,
    v_tilde_0=cvx_perf_params.v_tilde_0
)
slow_skf_param = filters.sKF_parameters(
    label='sKF_slow (Filter 2)',
    epsilon=cvx_perf_params.epsilon_2,
    var_eta=cvx_perf_params.var_eta,
    v_tilde_0=cvx_perf_params.v_tilde_0
)

baseline_measures, _ = mc.dask_MC_Simulations(
    N=N, NR=NR,
    environment_parameters=env_parameters,
    environment=environment,
    algorithms=[filters.sKF_algorithm, filters.sKF_algorithm],
    algorithms_parameters=[fast_skf_param, slow_skf_param],
    desired_metrics={'MSD': metrics.squared_deviation},
    h0=h0, num_workers=num_workers, num_chunks=num_chunks, seed=seed
)

# Plot curves in dB
plt.figure(figsize=(10, 6))

msd_best = exp.default_db_scale(grid_results['measures'][best_label]['performance'])
msd_worst = exp.default_db_scale(grid_results['measures'][worst_label]['performance'])
msd_median = exp.default_db_scale(grid_results['measures'][median_label]['performance'])
msd_fast = exp.default_db_scale(baseline_measures['sKF_fast (Filter 1)']['MSD'])
msd_slow = exp.default_db_scale(baseline_measures['sKF_slow (Filter 2)']['MSD'])

plt.plot(msd_best, color='crimson', lw=2.2, label=f"Optimal cvx_comb_sKF ({grid_results['best_metric_value']:.1f} dB*iter)")
plt.plot(msd_median, color='orange', lw=1.5, ls='--', label=f"Median cvx_comb_sKF ({c_grid[median_idx]:.1f} dB*iter)")
plt.plot(msd_worst, color='brown', lw=1.2, ls=':', label=f"Worst cvx_comb_sKF ({c_grid[worst_idx]:.1f} dB*iter)")
plt.plot(msd_fast, color='blue', lw=1.2, ls='-.', alpha=0.7, label=f"sKF Fast Baseline (eps1={grid_results['best_parameters']['epsilon_1']:.1e})")
plt.plot(msd_slow, color='green', lw=1.2, ls='-.', alpha=0.7, label=f"sKF Slow Baseline (eps2={cvx_perf_params.epsilon_2:.1e})")

plt.xlabel("Iteration", fontsize=11)
plt.ylabel("Normalized MSD [dB]", fontsize=11)
plt.title("MSD Learning Curves: Optimal cvx_comb_sKF vs. Baselines", fontsize=13)
plt.grid(True, alpha=0.3)
plt.legend(fontsize=9, loc='upper right')
plt.ylim([min(np.min(msd_slow), np.min(msd_best)) - 3, 5])
plt.tight_layout()
plt.show()

## 7. Interactive Heatmap with Dropdowns and Sliders

The interactive tool allows dynamically selecting:
- **X-Axis and Y-Axis** from the hyperparameter set.
- **Metric type**: Convergence Area or Steady-State MSD.
- **Sliders for fixed parameters**: Sliders are initialized to the optimal coordinates and allow exploring arbitrary 2D hyperparameter slices.

In [ ]:
interactive_widget = exp.interactive_grid_search_heatmap(grid_results)
interactive_widget